# Data pre-processing <br>

Audio and text files need to be pre-processed i.e. normalized, uniformly formatted, aligned and segmented to ensure smooth model training. <br>

In this notebook: <br>

0. Imports & Global Variables <br>
    0.1 Install and import <br>
    0.2 Custom functions <br>
    0.3 Global variables <br>

1. Text pre-processing <br>
    1.1 Text normalization (Lowercase, Punctuation revomal, Diacritic Normalization) <br>
    1.2 Splitting different stories contained in the same file in the rgn-ita folder <br>
    1.3 Removing titles from transcript when it is not said in the recording <br>
    1.4 Checking total number of stories (transcripts in txt format) <br>

2. Audio pre-processing <br>
    2.1 Audio resampling and normalization <br>
    2.2 Manual segmentation on Audacity <br>
    2.3 Cleaning filenames + Merging multi-part stories <br>
    2.4 Checking total number of audio stories (in wav format) <br>

3. STT (Speech-To-Text) Forced alignment and segmentation <br>
    3.1 Getting models ready <br>
    3.2 Creating Whisper chunks (STT alignment and segmentation) <br>
    3.3 Validation step <br>
    3.4 Creating longer Whisper chunks <br>

4. Splitting dataset <br>
    4.1 Calculating author' contribution to dataset (speech time, tokens, number of stories) <br>
    4.2 Calculating speakers' contribution to  dataset (speech time, tokens, number of stories) <br>
    4.3 Selecting speakers going into different dataset partitions <br>
    4.4 Dividing Whisper chunks into three partitions <br>

5. Dataset statistics <br>
    5.1 Calculating raw audio duration <br>
    5.2 Calculating segmented audio chunks' duration <br>
        5.2.1 Short Whisper chunks <br>
        5.2.2 Longer Whisper chunks <br>
    5.3 Calculating info about train, dev and test partitions <br>
        5.3.1 Shortseg dataset <br>
        5.3.2 Longseg dataset <br>

N.B. You can find more detailed explanations of what happens in each step at the beginning of the five sections. <br>

*N.B.2 Not ready to run out of the box: adjust local paths, model names, and checkpoint directories before execution.* <br>

______________________________________________________________________________________________________________________________________

STRUCTURE and CONTENTS of the local DATASET FOLDER ("Pre-processing_output"):
```text
Pre-processing_output/
│
├── longseg_dataset_split/
│   ├── dev/                  
│   ├── test/
│   └── train/ 
|
├── rgn_only/
│   ├── audio/                 # Standalone story WAVs (*.wav)
│   ├── longer_whisper_chunks/      # Longer audio snippets with paired transcripts
│   ├── text/                  # Standalone story texts (*.txt)
│   └── whisper_chunks         # Short audio snippets with paired transcripts
|
├── rgn-ita/
│   ├── audio/                 # Master normalized WAV files (multi-story)
│   ├── audio_split/           # Standalone audio stories in Romagnol (manually split) (*.wav)
│   ├── audio_split_old/       # (Deprecated) Old standalone audio stories in Italian and Romagnol (with lower cutoff frequency) (*.wav)
│   ├── longer_whisper_chunks/       # Longer audio snippets with paired transcripts 
│   ├── stories_split/         # Standalone story texts in Italian and in Romagnol (*.txt)
│   ├── text/                  # Master normalized files (multi-story)
│   └── whisper_chunks         # Short audio snippets with paired transcripts
│
├── shortseg_dataset_split/
│   ├── dev/                  
│   ├── test/
│   └── train/ 
│
├── statistics_data_graphs/
│   ├── info_authors/
│   │   ├── bar_stories_x_author.png
│   │   ├── bar_time_x_author.png
│   │   ├── bar_tokens_x_author.png
│   │   ├── info_authors_df.html
│   │   ├── pie_author_x_stories.png
│   │   ├── pie_author_x_time.png
│   │   └── pie_author_x_tokens.png
│   ├── info_longer_chunks/
│   │   ├── hist_durations_in_rgn_only_longer_whisper_chunks.png
│   │   ├── hist_durations_in_rgn_only_longer_whisper_chunks_rgn-ita_longer_whisper_chunks.png
|   |   ├── hist_durations_in_rgn-ita_longer_whisper_chunks.png
│   │   └── risorse_rgn.xlsx     # Summary excel sheet with list of chunks (txt-wav pairs) with author name and speaker name(s) 
│   ├── info_partitions_longseg/
│   │   ├── hist_durations_in_longseg_dataset_split_dev.png
│   │   ├── hist_durations_in_longseg_dataset_split_test.png
│   │   └── hist_durations_in_longseg_dataset_split_train.png
│   ├── info_partitions_shortseg/
│   │   ├── hist_durations_in_shortseg_dataset_split_dev.png
│   │   ├── hist_durations_in_shortseg_dataset_split_test.png
│   │   └── hist_durations_in_shortseg_dataset_split_train.png
│   ├── info_short_chunks/
│   │   ├── hist_durations_in_rgn_only_whisper_chunksv.png
│   │   ├── hist_durations_in_rgn_only_whisper_chunks_rgn-ita_whisper_chunks.png
|   |   ├── hist_durations_in_rgn-ita_whisper_chunks.png
│   │   └── risorse_rgn.xlsx     # Summary excel sheet with list of chunks (txt-wav pairs) with author name and speaker name(s) 
│   ├── info_speakers/
│   │   ├── bar_stories_x_speaker.png
│   │   ├── bar_time_x_speaker.png
│   │   ├── bar_tokens_x_speaker.png
│   │   ├── info_speakers_df.html
│   │   ├── pie_speaker_x_stories.png
│   │   ├── pie_speaker_x_time.png
│   │   ├── pie_speaker_x_tokens.png
│   │   └── speakers_in_partitions.csv
│   ├── raw_audio_time/
│   │   ├── hist_durations_in_rgn_only_audio.png
│   │   ├── hist_durations_in_rgn-ita_audio_split.png
│   │   └── hist_durations_in_rgn-ita_audio_split_rgn_only_audio.png
│   └── rgn-ita_riassunto.xlsx   # Excel sheet I used to check rgn-ita sub-folder.
│
├── files_to_validate.xlsx   # Random selection of files whose duration amounts to 15% of the total audio time in the dataset (41 minutes and 16.73 seconds, 58 files).
│
└── README.txt   # Description of the "pre-processing_output" folder and its contents + description of issues and how they were handled.
```

______________________________________________________________________________________________________________________________________

References: <br>
https://stackoverflow.com/questions/71764492/upload-download-files-from-onedrive-with-python-do-i-need-an-azure-account <br>
https://www.geeksforgeeks.org/data-analysis/preprocessing-the-audio-dataset/ <br>
https://medium.com/@jprebys/audio-file-analysis-with-librosa-4798988f7e31 <br>
https://www.youtube.com/watch?v=6nLx1Zqmkuw <br>
https://stackoverflow.com/a/72238956 <br>
https://note.nkmk.me/en/python-collections-counter/#how-to-use-collectionscounter <br>
https://stackoverflow.com/questions/62593913/plotting-a-pie-chart-out-of-a-dictionary <br>
https://www.geeksforgeeks.org/pandas/bar-plot-in-matplotlib/ <br>
https://stackoverflow.com/questions/775049/how-do-i-convert-seconds-to-hours-minutes-and-seconds <br>
https://stackoverflow.com/questions/1024847/how-can-i-add-new-keys-to-a-dictionary <br>
https://stackoverflow.com/questions/18837262/convert-python-dict-into-a-dataframe <br>
https://stackoverflow.com/questions/12555323/how-to-add-a-new-column-to-an-existing-dataframe <br>
https://pypi.org/project/tabulate/ <br>
https://www.geeksforgeeks.org/python/check-if-a-file-exists-in-python/ <br>
https://stackoverflow.com/questions/20167930/start-index-at-1-for-pandas-dataframe <br>
https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.to_html.html <br>
https://pynative.com/python-delete-lines-from-file/ <br>

## 0. Imports & Functions & Global Variables

### 0.1 Install and import

In [ ]:
! pip install faster-whisper
! pip install humanfriendly
! pip install librosa
! pip install natsort
! pip install openpyxl
! pip install pandas
! pip install stable-ts

In [ ]:
import collections
from collections import defaultdict
import csv
import difflib
import gc
import os
from pathlib import Path
import random
import re
import shutil
import subprocess
import warnings
import wave
import IPython.display as ipd
import librosa
import matplotlib.pyplot as plt
import numpy as np
import openpyxl
import pandas as pd
import soundfile as sf
import stable_whisper
import torch
from faster_whisper import WhisperModel, download_model
from humanfriendly import format_timespan
from natsort import natsorted, os_sorted
from scipy.signal import butter, filtfilt
from tqdm.auto import tqdm

### 0.2 Custom functions

In [ ]:
def print_path_contents(path):
    """
    Returns the contents of a path, ensuring the access to the folders is clear and that files are visible.
    """
    items = os_sorted(os.listdir(path))
    return items


def preprocess_text(input_dir, text_folders, output_dir):
    """
    Normalizes apostrophes, converts slashes to spaces, preserves intra-word hyphens
    (e.g., s-ciare), lowercases, strips residual punctuation, and preserves
    all newlines and blank lines for story splitting.
    """
    for sub in text_folders:
        in_dir = input_dir / sub
        out_dir = output_dir / sub
        out_dir.mkdir(parents=True, exist_ok=True)
        for file in tqdm(list(in_dir.glob("*.txt"))):
            text = file.read_text(encoding="utf-8")
            text = text.replace("’", "'").replace("`", "'").replace("‘", "'") # Normalize curly/slanted apostrophes
            text = text.replace("/", " ") # Replace slashes with a space so 'leader/lédar' becomes 'leader lédar'
            text = text.lower()    # Lowercase
            text = re.sub(r"(?<!\w)-|-(?!\w)", " ", text) # Strip only standalone hyphens, not those in words.
            text = re.sub(r"[^\w\s'-]", "", text) # Strip punctuation (except letters, digits, spaces, newlines, ', and -)
            text = re.sub(r"[^\S\n\r]+", " ", text) # Collapse horizontal spaces (spaces/tabs) while PRESERVING newlines & blank lines
            text = re.sub(r"^[^\S\n\r]+|[^\S\n\r]+$", "", text, flags=re.MULTILINE).strip()
            (out_dir / file.name).write_text(text, encoding="utf-8") # Save

def split_text_stories(input_dir, name_input_folder, output_dir, name_output_folder): 
    """
    Extracts and save narrative units into original documents into standalone txt files.
    """
    input_dir = input_dir / name_input_folder
    split_output_dir = output_dir / name_output_folder
    split_output_dir.mkdir(parents=True, exist_ok=True)

    files = sorted(list(input_dir.glob("*.txt")))

    for file in tqdm(files, desc="Splitting stories"):
        content = file.read_text(encoding="utf-8").strip()
        blocks = [b.strip() for b in re.split(r"\n\s*\n", content) if b.strip()] # Split into blocks separated by one or more blank lines
        story_idx = 1
        for i in range(0, len(blocks), 2):  # Pair title and text
            if i + 1 < len(blocks):
                title = blocks[i]
                body = blocks[i + 1]
                story_text = f"{title}\n\n{body}"
                out_name = f"{file.stem}_st{story_idx}.txt"
                (split_output_dir / out_name).write_text(story_text, encoding="utf-8")
                story_idx += 1


def remove_titles(list_of_files, text_dir):
    """
    Removes titles (first line of the transcript)
    """
    for f in list_of_files:
        f_path = text_dir / \
            f"{f}.txt" if not str(f).endswith(".txt") else text_dir / f
        with open(file=f_path, mode="r+", encoding="utf-8") as txt:
            lines = txt.readlines()
            txt.seek(0) # Go to first line
            txt.truncate() # Truncate it
            txt.writelines(lines[2:]) # start writing lines except the first line
    print("All done. Titles were removed from the specified list of files.")

def print_stats(name_dir, name_folder, abbr_lang_1=None, abbr_lang_2=None):
    """
    Counts number of files (text or audio) in a certain folder.
    If a language is specified it will count the occurrences for resources in that language.
    """
    input_dir = name_dir / name_folder
    files = [f for f in input_dir.glob("*") if f.is_file()]

    print(f"\n--- Summary files in '{name_folder}' ---")
    
    if abbr_lang_1 or abbr_lang_2:  # If languages are passed as arguments:
        tot_lang_1 = 0
        tot_lang_2 = 0
        for file in files:
            if abbr_lang_1 and file.name.startswith(abbr_lang_1):
                tot_lang_1 += 1
            elif abbr_lang_2 and file.name.startswith(abbr_lang_2):
                tot_lang_2 += 1
        if abbr_lang_1:
            print(f"Total {abbr_lang_1} stories : {tot_lang_1}")
        if abbr_lang_2:
            print(f"Total {abbr_lang_2} stories : {tot_lang_2}")
    print(f"Total stories overall: {len(files)}")

def butter_lowpass_filter(data, cutoff_freq, sample_rate, order=4):
    """
    Filters noisy recordings by removing high-frequency hiss/noise above a certain threshold.
    """
    nyquist = 0.5 * sample_rate
    normal_cutoff = cutoff_freq / nyquist
    b, a = butter(order, normal_cutoff, btype='low', analog=False)
    filtered_data = filtfilt(b, a, data)
    print(f"Filtered audio shape: {filtered_data.shape}")
    return filtered_data

def peak_amplitude_normalizer(audio, target_peak=0.95):
    """
    Scales the entire audio waveform uniformly so that its 
    loudest peak reaches `target_peak` (default 95% of max amplitude, 
    leaving headroom to prevent clipping).
    """
    peak = np.max(np.abs(audio))
    if peak > 0:
        return audio * (target_peak / peak)
    return audio

def preprocess_audio(input_dir, input_folders, output_dir):
    """
    Standardizes all audio files to single-channel (mono) WAV format at 16,000 Hz (16 kHz)
    """
    CUTOFF_FREQ = 7000  
    for sub in input_folders:
        in_dir = input_dir / sub
        out_dir = output_dir / sub
        out_dir.mkdir(parents=True, exist_ok=True)
        mp3_files = list(in_dir.glob("*.mp3"))
        for mp3 in tqdm(mp3_files, desc=f"Processing {sub}"):
            audio, sr = librosa.load(str(mp3), sr=16000, mono=True) # Load MP3 as 16 kHz mono
            if audio.size == 0:
                    print(f"Skipping empty audio: {mp3.name}")
                    continue
            audio_filtered = butter_lowpass_filter(audio, cutoff_freq=CUTOFF_FREQ, sample_rate=sr, order=4) # Low-pass filter ()
            audio_norm = peak_amplitude_normalizer(audio_filtered, target_peak=0.95)  # Peak volume normalization 
            out_wav = out_dir / f"{mp3.stem}.wav" # Save 16 kHz WAV
            sf.write(str(out_wav), audio_norm, sr)
    print("All done! Audio pre-processing was successful.")

def clean_filenames(audio_dir):
    """
    Cleans filenames from any extra whitespaces.
    """
    for wav in audio_dir.glob("*.wav"):
        if " " in wav.name:
            clean_name = wav.name.replace(" ", "")
            wav.rename(wav.parent / clean_name)
            print(f"Fixed: {wav.name} -> {clean_name}")

def merge_multi_part_audio(input_dir, silence_duration=0.2):
    """
    Merges multiple audio clips of subsequent parts in the same story together.
    """
    multi_part_stories = []
    for file in input_dir.rglob("rgn*"):
        if re.search(r"st[0-9]+[a-z]+\.wav$", file.name):
            multi_part_stories.append(file)

    processed_stories = set()

    sample_rate = 16000
    num_silence_samples = int(sample_rate * silence_duration)
    silence_padding = np.zeros(num_silence_samples, dtype=np.float32)

    for f in multi_part_stories:
        # Extract everything before the part letter 'a/b/c' (e.g., 'rgn_bal_1_st4a.wav' -> 'rgn_bal_1_st4')
        match_prefix = re.match(r"^(.*st\d+)[a-z]+\.wav$", f.name)
        if not match_prefix:
            continue

        story_prefix = match_prefix.group(1)

        if story_prefix in processed_stories:
            continue

        # Strictly match files that end with <story_prefix> + <letter> + .wav
        matches = [
            file
            for file in multi_part_stories
            if re.match(rf"^{re.escape(story_prefix)}[a-z]+\.wav$", file.name)
        ]

        print(f"Found matches for {story_prefix}: {[m.name for m in matches]}")
        matches.sort(key=lambda x: x.name)
        audio_segments = []
        for i, match_file in enumerate(matches):
            data, _ = sf.read(match_file)
            if i > 0 and silence_duration > 0:
                audio_segments.append(silence_padding)
            audio_segments.append(data)
        combined_audio = np.concatenate(audio_segments, axis=0)
        merged_file_path = f.parent / f"{story_prefix}.wav" # Saves as C:/.../audio_split/rgn_bal_1_st4.wav
        sf.write(merged_file_path, combined_audio, sample_rate)
        for match_file in matches:
            match_file.unlink()
            print(f"Deleted: {match_file.name}")
        processed_stories.add(story_prefix)

def env_check():
    """
    Checks the environment where notebook is being run.
    """
    is_cuda = torch.cuda.is_available()
    device = "cuda" if is_cuda else "cpu"
    compute_type = "float16" if is_cuda else "int8"
    model_size = "large-v3" if is_cuda else "small"
    print(f"Target Environment: {device.upper()} | Model: {model_size} ({compute_type})")
    return is_cuda, device, compute_type, model_size


def setup_model(model_size: str, base_dir: str | Path) -> Path:
    """
    Ensures CTranslate2 weights exist inside the specified base directory.
    Downloads them if they do not exist.
    """
    base_dir = Path(base_dir).resolve() # Convert input to a Path object and resolve absolute path
    model_dir = base_dir / f"whisper_{model_size}_weights"
    model_bin_file = model_dir / "model.bin"
    if not model_bin_file.is_file():
        print(f"Downloading CTranslate2 '{model_size}' weights to disk...")
        download_model(model_size, output_dir=str(model_dir))  # pass a string to output_dir because download_models expects a string path
    else:
        print(f"Found existing weights for '{model_size}'.")
    print(f"Model ready at: {model_dir}")
    return model_dir

def info_durations(dir_1, output_dir, over_time=None, overview=None, totals=None, graph=None, dir_2=None):
    """
    Summarizes duration statistics for .wav files in one or two directories.
    Args:
        dir_1 (Path): Primary directory containing .wav files.
        output_dir (Path): Output directory for saving generated figures.
        over_time (float, optional): Custom threshold in seconds to print specific file counts.
        overview (bool, optional): Print fixed tiered duration breakdowns (5, 10, 15, 20s).
        totals (bool, optional): Print total runtime, mean, and std deviation.
        graph (bool, optional): Display a duration histogram.
        dir_2 (Path, optional): Optional second directory to pool with dir_1.
    """
    durations_1 = [sf.info(f).duration for f in dir_1.glob("**/*.wav")]
    arr_1 = np.array(durations_1)
    two_last_names_1 = ["_".join(dir_1.parts[-2:]) if len(dir_1.parts) >= 2 else str(dir_1)]
    len_dir_1 = len(arr_1)
    total_time_1 = float(arr_1.sum())
    if dir_2:
        durations_2 = [sf.info(f).duration for f in dir_2.glob("**/*.wav")]
        arr_2 = np.array(durations_2)
        arr_both_dirs = np.concatenate((arr_1, arr_2))
        two_last_names_2 = [
            "_".join(dir_2.parts[-2:]) if len(dir_2.parts) >= 2 else str(dir_2)
        ]
        len_both_dirs = len_dir_1 + len(arr_2)
        if overview:
            print(f"Files found in '{two_last_names_1[0]}' and '{two_last_names_2[0]}': {len_both_dirs}.\nBreakdown of durations:")
            for threshold in [5, 10, 15, 20]:
                count = (arr_both_dirs > threshold).sum()
                pct = (count / len_both_dirs) if len_both_dirs > 0 else 0
                print(f"Files over {threshold} s: {count} ({pct:.0%})")
            print()

        # Only print single-threshold count if over_time was explicitly provided
        if over_time is not None:
            count = (arr_both_dirs > over_time).sum()
            pct = (count / len_both_dirs) if len_both_dirs > 0 else 0
            print(f"Files over {over_time} s in '{two_last_names_1[0]}' and '{two_last_names_2[0]}': {count} out of {len_both_dirs} ({pct:.0%}).\n")

        if totals:
            total_time_2 = float(arr_2.sum())
            total_time_dataset = format_timespan(total_time_1 + total_time_2)
            mean_both = np.mean(arr_both_dirs)
            std_both = np.std(arr_both_dirs)
            print(f"Total audio time in directory '{two_last_names_1[0]}' + '{two_last_names_2[0]}': {total_time_dataset}.")
            print(f"Mean audio time and standard deviation: {mean_both:.2f} ± {std_both:.2f} (s).\n")

        if graph:
            plt.hist(arr_both_dirs, bins=20, color="skyblue", edgecolor="black")
            plt.xlabel("Duration (s)")
            plt.ylabel("Frequency")
            plt.title(f"Distribution of durations in '{two_last_names_1[0]}' and '{two_last_names_2[0]}'")
            plt.savefig(output_dir / f"hist_durations_in_{two_last_names_1[0]}_{two_last_names_2[0]}.png")
            print(f"Histogram saved as image in {output_dir}!")
            plt.show()
    else:
        if overview:
            print(f"Files found in '{two_last_names_1[0]}': {len_dir_1}.\nBreakdown of durations:")
            for threshold in [5, 10, 15, 20]:
                count = (arr_1 > threshold).sum()
                pct = (count / len_dir_1) if len_dir_1 > 0 else 0
                print(f"Files over {threshold} s: {count} ({pct:.0%})")
            print(f"Total audio time in '{two_last_names_1[0]}': {total_time_1 / 3600:.2f} hours ({total_time_1:.0f} seconds)\n")

        # Only print single-threshold count if over_time was explicitly provided
        if over_time is not None:
            count = (arr_1 > over_time).sum()
            pct = (count / len_dir_1) if len_dir_1 > 0 else 0
            print(f"Files over {over_time} s in '{two_last_names_1[0]}': {count} out of {len_dir_1} ({pct:.0%}).\n")

        if totals:
            mean_1 = np.mean(arr_1)
            std_1 = np.std(arr_1)
            print(f"Total audio time in directory '{two_last_names_1[0]}': {total_time_1 / 3600:.2f} hours ({total_time_1:.0f} seconds).")
            print(f"Mean audio time and standard deviation: {mean_1:.2f} ± {std_1:.2f} (s).\n")

        if graph:
            plt.hist(arr_1, bins=20, color="darkorchid", edgecolor="black")
            plt.xlabel("Duration (s)")
            plt.ylabel("Frequency")
            plt.title(f"Distribution of durations in '{two_last_names_1[0]}'")
            plt.savefig(output_dir / f"hist_durations_in_{two_last_names_1[0]}.png")
            print(f"Histogram saved as image in {output_dir}!")
            plt.show()


def load_stable_whisper(model_id, device, compute_type):
    # Pass dynamic CPU threads when running on CPU, or let CUDA handle itself
    threads = 4 if device == "cpu" else 0

    model = stable_whisper.load_faster_whisper(
        str(model_id),  # <--- Convert Path object to string here
        device=device,
        compute_type=compute_type,
        cpu_threads=threads,
    )
    print("Done, model ready for use.")
    return model


def align_and_chunk(model, part_dataset, single_file=None):
    audio_dir, text_dir, out_dir = map(Path, DATASETS[part_dataset])
    out_dir.mkdir(parents=True, exist_ok=True)

    # Target files
    if single_file:
        stem = Path(single_file).stem.strip()
        wav_files = [f for f in audio_dir.glob(
            "*.wav") if f.stem.strip() == stem]
        if not wav_files:
            raise FileNotFoundError(f"'{stem}.wav' not found in {audio_dir}")
    else:
        wav_files = sorted(audio_dir.glob("*.wav"))

    meta = []

    # Main processing loop
    pbar = tqdm(wav_files, desc=f"Aligning [{part_dataset}]")
    for wav in pbar:
        pbar.set_postfix(file=wav.name)
        txt = text_dir / f"{wav.stem}.txt"
        if not txt.exists():
            continue

        audio, sr = sf.read(str(wav), dtype="float32")
        total_dur = len(audio) / sr
        lines = [
            line.strip()
            for line in txt.read_text(encoding="utf-8").splitlines()
            if line.strip()
        ]
        if not lines:
            continue

        alignment_failed = False

        # Attempt standard forced alignment
        with warnings.catch_warnings(record=True) as captured:
            warnings.simplefilter("always")
            res = model.align(
                audio,
                "\n".join(lines),
                language="it",
                suppress_silence=False,
                vad=False,
                original_split=True,
                failure_threshold=None,
                fast_mode=False,
                word_dur_factor=2.0,
            )
            if len(captured) > 0:
                alignment_failed = True

        # Sanity check: verify segment count and check for collapsed timestamps
        if not alignment_failed:
            if not hasattr(res, "segments") or len(res.segments) != len(lines):
                alignment_failed = True
            else:
                zero_spans = sum(1 for s in res.segments if (
                    s.end - s.start) <= 0.05)
                tail_collapsed = (
                    len(res.segments) >= 2
                    and abs(res.segments[-1].start - res.segments[-2].start) < 0.05
                )
                if zero_spans > 0 or tail_collapsed:
                    alignment_failed = True

        time_pairs = []

        # Branch A: Alignment succeeded directly
        if not alignment_failed:
            time_pairs = [(s.start, s.end) for s in res.segments]

        # Branch B: Fallback to acoustic speech pause segments
        else:
            print(
                f"\n[Fallback] Alignment issue on {wav.name} -> Deriving pause-aware acoustic boundaries")
            free_res = model.transcribe(
                audio, language="it", suppress_silence=False)
            speech_segs = [s for s in free_res.segments if s.text.strip()]

            # Subdivide the longest segments until count matches lines
            while len(speech_segs) < len(lines):
                longest_idx = max(
                    range(len(speech_segs)),
                    key=lambda i: speech_segs[i].end - speech_segs[i].start,
                )
                target = speech_segs.pop(longest_idx)
                mid = (target.start + target.end) / 2.0

                seg_a = type("Seg", (), {"start": target.start, "end": mid})()
                seg_b = type("Seg", (), {"start": mid, "end": target.end})()
                speech_segs.insert(longest_idx, seg_b)
                speech_segs.insert(longest_idx, seg_a)

            # Build non-overlapping boundary pairs that preserve trailing silence
            for i in range(len(lines)):
                s_curr = speech_segs[i]
                start_t = s_curr.start
                if i + 1 < len(speech_segs):
                    s_next = speech_segs[i + 1]
                    end_t = min(
                        s_next.start - 0.04,
                        s_curr.end +
                        max(0.0, (s_next.start - s_curr.end) * 0.6),
                    )
                else:
                    end_t = min(total_dur, s_curr.end + 0.35)
                time_pairs.append((start_t, end_t))

        if not time_pairs:
            continue

        # Slice and export chunks
        for i, ((t_start, t_end), line) in enumerate(zip(time_pairs, lines), 1):
            is_last = (i == len(time_pairs))
            prev_end = time_pairs[i - 2][1] if i > 1 else 0.0

            # Safe leading boundary clamped to previous segment's end
            t0 = max(0.0, max(prev_end, t_start - 0.15))

            if is_last:
                # Extend the final segment all the way to the end of the recording
                t1 = max(t0 + 0.5, total_dur - 0.05)
            else:
                next_start = time_pairs[i][0]
                # Clamp tail to avoid bleeding into next speech onset
                t1 = min(total_dur, min(next_start - 0.02, t_end + 0.25))

            if t1 <= t0:
                t1 = min(total_dur, t0 + 0.30)

            chunk = audio[int(t0 * sr): int(t1 * sr)]
            if len(chunk) == 0:
                continue

            stem = f"{wav.stem}_seg{i:03d}"
            sf.write(str(out_dir / f"{stem}.wav"), chunk, sr, subtype="PCM_16")
            (out_dir / f"{stem}.txt").write_text(line, encoding="utf-8")

            meta.append({
                "audio_file": f"{stem}.wav",
                "text_file": f"{stem}.txt",
                "duration": round(len(chunk) / sr, 2),
                "text": line,
            })

    # Save metadata.csv
    if meta:
        csv_file = out_dir / "metadata.csv"
        write_header = not csv_file.exists() or csv_file.stat().st_size == 0
        with open(csv_file, "a", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(
                f, fieldnames=["audio_file", "text_file", "duration", "text"]
            )
            if write_header:
                writer.writeheader()
            writer.writerows(meta)

    print(f"Done! Created {len(meta)} chunks in: {out_dir}")


def info_authors(dir_audio_1, dir_audio_2, dir_txt_1, dir_txt_2, output_dir, totals=None, pie=None, bar=None):
    """
    What this function does: 
    - creates a table with data about how each author contributes to dataset in terms of speech time, total of tokens, number of stories.
    - saves an html with the table containing all this info.
    - can print and save pie charts and bar charts to visualize data about Romagnol authors.
    Note: Speech time is calculated on raw time (unsegmented audio files).
    """
    files_1_audio = list(dir_audio_1.glob("*.wav"))
    files_2_audio = list(dir_audio_2.glob("*.wav"))
    all_files_audio = files_1_audio + files_2_audio

    files_1_txt = list(dir_txt_1.glob("rgn*.txt"))
    files_2_txt = list(dir_txt_2.glob("rgn*.txt"))
    all_files_txt = files_1_txt + files_2_txt

    author_names = []
    for s in files_1_audio:
        author_s = re.findall(r"_[a-z]{3,4}_", str(s))
        if author_s:
            author_names.append(author_s[0].strip("_"))

    for p in files_2_audio:
        author_p = re.findall(r"_[a-z]{3,4}_", str(p))
        if author_p:
            author_names.append(author_p[0].strip("_"))
    len_tot = len(author_names)

    author_freq_n_st = collections.Counter(author_names)
    names = list(author_freq_n_st.keys())
    n_stories_for_author = list(author_freq_n_st.values())
        
    tot_time_authors = {}
    tot_tokens_authors = {}

    for name in names:
        tot_time_name = 0.0
        tot_tokens_name = 0

        # Audio durations
        for f in all_files_audio:
            if f"_{name}_" in f.name:
                tot_time_name += sf.info(f).duration
        tot_time_authors[name] = round(tot_time_name, 2)

        # Token counts
        for f in all_files_txt:
            if f"_{name}_" in f.name:
                text = f.read_text(encoding="utf-8")
                tokens = text.split()
                tot_tokens_name += len(tokens)
        tot_tokens_authors[name] = tot_tokens_name

    tot_time_dataset_seconds = sum(tot_time_authors.values())
    tot_tokens_dataset = sum(tot_tokens_authors.values())

    pct_time_authors = {
        name: round((dur / tot_time_dataset_seconds) * 100,
                    2) if tot_time_dataset_seconds > 0 else 0.0
        for name, dur in tot_time_authors.items()
    }

    pct_tok_authors = {
        name: (
            round((tok / tot_tokens_dataset) * 100, 2)
            if tot_tokens_dataset > 0
            else 0.0
        )
        for name, tok in tot_tokens_authors.items()
    }

    pct_st_authors = {
        name: round((count / len_tot) * 100, 2) if len_tot > 0 else 0.0
        for name, count in author_freq_n_st.items()
    }
    
    if totals:
        print(f"Total number of stories in Romagnol: {len_tot}")
        print(f"Total number of authors in the dataset: {len(author_freq_n_st)}")
        tot_time_dataset_reg_format = format_timespan(tot_time_dataset_seconds)
        print(f"Total audio time in dataset: {tot_time_dataset_reg_format}")
        print(f"Total tokens in dataset: {tot_tokens_dataset}\n")
    
    df = pd.DataFrame(tot_time_authors.items(), columns=['Authors', 'Audio time (s)'])
    df_time_percentages = pd.DataFrame(pct_time_authors.values())
    df_tok = pd.DataFrame(tot_tokens_authors.values())
    df_tok_percentages = pd.DataFrame(pct_tok_authors.values())
    df_n_stories = pd.DataFrame(n_stories_for_author)
    df_st_percentages = pd.DataFrame(pct_st_authors.values())
    df['% tot time'] = df_time_percentages.values
    df['Tokens'] = df_tok.values
    df['% tot tokens'] = df_tok_percentages.values
    df['N stories'] = df_n_stories.values
    df['% tot stories'] = df_st_percentages.values
    df.index += 1
    print(df)
    df.to_html(output_dir / 'info_authors_df.html')
    print(f"\nAn html version of this dataframe was saved in {output_dir}!")

    names_dict_time = list(tot_time_authors.keys())
    time_dict_time = list(tot_time_authors.values())

    names_dict_tokens = list(tot_tokens_authors.keys())
    tokens_dict_tokens = list(tot_tokens_authors.values())

    if pie:
        plt.pie(n_stories_for_author, labels=names,
                labeldistance=1.0, rotatelabels=True)
        plt.title(f"Author x number of stories")
        plt.axis('equal')
        plt.savefig(output_dir / 'pie_author_x_stories')
        plt.show()

        plt.pie(time_dict_time, labels=names_dict_time, labeldistance=1.0, rotatelabels=True)
        plt.title(f"Author x total audio time")
        plt.axis('equal')
        plt.savefig(output_dir / 'pie_author_x_time')
        plt.show()

        plt.pie(tokens_dict_tokens, labels=names_dict_tokens, labeldistance=1.0, rotatelabels=True)
        plt.title(f"Author x total tokens")
        plt.axis('equal')
        plt.savefig(output_dir / 'pie_author_x_tokens')
        plt.show()

    if bar:
        plt.barh(names, n_stories_for_author, color='darksalmon', height=0.5)
        plt.title('Number of stories for each author')
        plt.xlabel('Number of stories')
        plt.ylabel('Authors')
        plt.savefig(output_dir / 'bar_stories_x_author')
        plt.show()

        plt.barh(names_dict_time, time_dict_time, color='plum', height=0.5)
        plt.title('Total audio time for each author')
        plt.xlabel('Total time (seconds)')
        plt.ylabel('Authors')
        plt.savefig(output_dir / 'bar_time_x_author')
        plt.show()

        plt.barh(names_dict_tokens, tokens_dict_tokens, color='forestgreen', height=0.5)
        plt.title('Total tokens for each author')
        plt.xlabel('Total tokens')
        plt.ylabel('Authors')
        plt.savefig(output_dir / 'bar_tokens_x_author')
        plt.show()


def info_speakers(dir_audio_1, dir_audio_2, dir_txt_1, dir_txt_2, output_dir, dict_speakers, totals=None, pie=None, bar=None):
    """
    What this function does: 
    - creates a table with data about how each speaker contributes to dataset in terms of speech time, total of tokens, number of stories.
    - saves an html with the table containing all this info.
    - can print and save pie charts and bar charts to visualize data about Romagnol speakers.
    Note: Speech time is calculated on raw time (unsegmented audio files).
    """
    files_1_audio = list(dir_audio_1.glob("*.wav"))
    files_2_audio = list(dir_audio_2.glob("*.wav"))
    all_files_audio = files_1_audio + files_2_audio

    files_1_txt = list(dir_txt_1.glob("rgn*.txt"))
    files_2_txt = list(dir_txt_2.glob("rgn*.txt"))
    all_files_txt = files_1_txt + files_2_txt

    author_names = []
    for s in files_1_audio:
        author_s = re.findall(r"_[a-z]{3,4}_", str(s))
        if author_s:
            author_names.append(author_s[0].strip("_"))

    for p in files_2_audio:
        author_p = re.findall(r"_[a-z]{3,4}_", str(p))
        if author_p:
            author_names.append(author_p[0].strip("_"))
    len_tot = len(author_names)

    author_freq_n_st = collections.Counter(author_names)
    
    # Map authors to speakers (handling slash-separated names)
    speaker_time_authors = collections.defaultdict(float)
    speaker_tokens_authors = collections.defaultdict(int)
    speaker_freq_n_st = collections.defaultdict(float)

    for author, count in author_freq_n_st.items():
        if author in dict_speakers:
            raw_spk = dict_speakers[author]
            speakers = [s.strip() for s in raw_spk.split("/") if s.strip()]
            num_spk = len(speakers)

            tot_time_name = 0.0
            tot_tokens_name = 0

            # Audio durations (original loop structure)
            for f in all_files_audio:
                if f"_{author}_" in f.name:
                    tot_time_name += sf.info(f).duration

            # Token counts (original loop structure)
            for f in all_files_txt:
                if f"_{author}_" in f.name:
                    text = f.read_text(encoding="utf-8")
                    tokens = text.split()
                    tot_tokens_name += len(tokens)

            # Distribute metrics among speakers
            for spk in speakers:
                speaker_time_authors[spk] += tot_time_name / num_spk
                speaker_tokens_authors[spk] += tot_tokens_name / num_spk
                speaker_freq_n_st[spk] += count / num_spk

    # Round aggregated values
    tot_time_authors = {spk: round(t, 2)
                        for spk, t in speaker_time_authors.items()}
    tot_tokens_authors = {spk: round(tok)
                          for spk, tok in speaker_tokens_authors.items()}
    speaker_freq_n_st = {spk: round(cnt, 2)
                         for spk, cnt in speaker_freq_n_st.items()}

    names = list(tot_time_authors.keys())
    n_stories_for_author = [speaker_freq_n_st[name] for name in names]

    tot_time_dataset_seconds = sum(tot_time_authors.values())
    tot_tokens_dataset = sum(tot_tokens_authors.values())

    pct_time_authors = {
        name: round((dur / tot_time_dataset_seconds) * 100,
                    2) if tot_time_dataset_seconds > 0 else 0.0
        for name, dur in tot_time_authors.items()
    }

    pct_tok_authors = {
        name: (
            round((tok / tot_tokens_dataset) * 100, 2)
            if tot_tokens_dataset > 0
            else 0.0
        )
        for name, tok in tot_tokens_authors.items()
    }

    pct_st_authors = {
        name: round((count / len_tot) * 100, 2) if len_tot > 0 else 0.0
        for name, count in speaker_freq_n_st.items()
    }

    if totals:
        print(f"Total number of stories in Romagnol: {len_tot}")
        print(f"Total number of speakers in the dataset: {len(names)}")
        tot_time_dataset_reg_format = format_timespan(tot_time_dataset_seconds)
        print(f"Total audio time in dataset: {tot_time_dataset_reg_format}")
        print(f"Total tokens in dataset: {tot_tokens_dataset}\n")

    df = pd.DataFrame(tot_time_authors.items(), columns=[
                      'Speakers', 'Audio time (s)'])
    df['% tot time'] = pct_time_authors.values()
    df['Tokens'] = pct_tok_authors.values()      # mapped to total tokens below
    df['Tokens'] = tot_tokens_authors.values()
    df['% tot tokens'] = pct_tok_authors.values()
    df['N stories'] = n_stories_for_author
    df['% tot stories'] = pct_st_authors.values()

    # --- Sort descending by Audio time ---
    df = df.sort_values(by='Audio time (s)', ascending=False)

    # Reset index so rank starts from 1 to N after sorting
    df = df.reset_index(drop=True)
    df.index += 1

    print(df)
    df.to_html(output_dir / 'info_speakers_df.html')
    print(f"\nAn html version of this dataframe was saved in {output_dir}!")

    names_dict_time = list(tot_time_authors.keys())
    time_dict_time = list(tot_time_authors.values())

    names_dict_tokens = list(tot_tokens_authors.keys())
    tokens_dict_tokens = list(tot_tokens_authors.values())

    if pie:
        plt.pie(n_stories_for_author, labels=names,
                labeldistance=1.0, rotatelabels=True)
        plt.title(f"Speaker x number of stories")
        plt.axis('equal')
        plt.savefig(output_dir / 'pie_speaker_x_stories')
        plt.show()

        plt.pie(time_dict_time, labels=names_dict_time,
                labeldistance=1.0, rotatelabels=True)
        plt.title(f"Speaker x total audio time")
        plt.axis('equal')
        plt.savefig(output_dir / 'pie_speaker_x_time')
        plt.show()

        plt.pie(tokens_dict_tokens, labels=names_dict_tokens,
                labeldistance=1.0, rotatelabels=True)
        plt.title(f"Speaker x total tokens")
        plt.axis('equal')
        plt.savefig(output_dir / 'pie_speaker_x_tokens')
        plt.show()

    if bar:
        plt.barh(names, n_stories_for_author, color='darksalmon', height=0.5)
        plt.title('Number of stories for each speaker')
        plt.xlabel('Number of stories')
        plt.ylabel('Speakers')
        plt.savefig(output_dir / 'bar_stories_x_speaker')
        plt.show()

        plt.barh(names_dict_time, time_dict_time, color='plum', height=0.5)
        plt.title('Total audio time for each speaker')
        plt.xlabel('Total time (seconds)')
        plt.ylabel('Speakers')
        plt.savefig(output_dir / 'bar_time_x_speaker')
        plt.show()

        plt.barh(names_dict_tokens, tokens_dict_tokens,
                 color='forestgreen', height=0.5)
        plt.title('Total tokens for each speaker')
        plt.xlabel('Total tokens')
        plt.ylabel('Speakers')
        plt.savefig(output_dir / 'bar_tokens_x_speaker')
        plt.show()


def export_file_speaker_mapping(dir_rgnonly_chunks, dir_rgnita_chunks, dict_speakers, output_excel_path):
    """
    Scans chunk directories for files, extracts author code using regex (_[a-z]{3,4}_),
    looks up corresponding speakers from dict_speakers, checks for .txt and .wav presence,
    and exports a naturally sorted Excel report.
    """
    output_excel_path = Path(output_excel_path)
    
    # Resolve output file path
    if output_excel_path.is_dir():
        output_file = output_excel_path / "file_author_speaker_mapping.xlsx"
    elif output_excel_path.suffix.lower() == ".xlsx":
        output_file = output_excel_path
    else:
        output_file = output_excel_path.with_suffix(".xlsx")

    output_file.parent.mkdir(parents=True, exist_ok=True) # Ensure parent directory exists

    author_pattern = re.compile(r"_[a-z]{3,4}_")

    # Collect all audio and text files from folders with chunks (both .txt and .wav)
    files_rgnonly = list(dir_rgnonly_chunks.glob("*.wav")) + list(dir_rgnonly_chunks.glob("*.txt"))
    files_rgnita = list(dir_rgnita_chunks.glob("*.wav")) + list(dir_rgnita_chunks.glob("*.txt"))
    all_chunk_files = files_rgnonly + files_rgnita

    # Map file stem of paired transcripts and recordings: e.g. {'chunk_1': {'.wav', '.txt'}})
    file_extensions_map = {}
    for f in all_chunk_files:
        stem = f.stem
        if stem not in file_extensions_map:
            file_extensions_map[stem] = set()
        file_extensions_map[stem].add(f.suffix.lower())

    # Extract unique base filenames and sort naturally
    sorted_file_stems = natsorted(list(file_extensions_map.keys()))

    records = []

    # Process each unique file stem
    for file_stem in sorted_file_stems:
        match = author_pattern.search(file_stem)
        
        if match:
            author = match.group(0).strip("_")
            
            if author in dict_speakers:
                raw_spk = dict_speakers[author]
                speakers = ", ".join([s.strip() for s in raw_spk.split("/") if s.strip()])
            else:
                speakers = "Unknown / Not in dict"
        else:
            author = "Unknown"
            speakers = "Unknown"

        exts = file_extensions_map[file_stem]
        txt_present = ".txt" in exts
        wav_present = ".wav" in exts

        records.append({
            "File Name": file_stem,
            "Author": author,
            "Speakers": speakers,
            "txt_file": txt_present,
            "wav_file": wav_present
        })

    # Create DataFrame and write/overwrite Excel file
    df = pd.DataFrame(records)
    df.to_excel(output_file, index=False, engine="openpyxl")
    
    print(f"Excel report saved: {output_file}")
    print(f"Total unique chunk files indexed: {len(df)}")
    
    return df


def files_to_validate(dir1, dir2, percentage, output_path):
    """
    This function creates an xlsx list of files whose Whisper chunks will need to be manually validated.
    Their durations amount to the percentage specified by the user (recommended: 15-20% of the total time in the dataset).
    """
    output_file = Path(output_path)
    # Stop if file already exists
    if output_file.exists():
        print(f"File already exists: {output_file}. Skipping execution.")
        return
    # Read files, durations, and second-to-last folder names
    files = []
    for directory in [dir1, dir2]:
        folder = Path(directory).resolve()
        parent_dir_name = folder.parent.name  # Second-to-last directory name
        for audio_path in folder.iterdir():
            dur = sf.info(str(audio_path)).duration
            files.append((audio_path.stem, parent_dir_name, dur))

    # Calculate target duration
    total_time = sum(dur for _, _, dur in files)
    target_time = total_time * (percentage / 100)
    print(f"Necessary to check {format_timespan(target_time)}.")

    # Randomize order
    random.shuffle(files)

    # Collect filenames until the target duration is reached
    selected_files = []
    current_time = 0.0

    for fname, dname, dur in files:
        selected_files.append((fname, dname))
        current_time += dur
        if current_time >= target_time:
            break
    print(f"You will need to check {len(selected_files)} files.")
    
    # Create Excel workbook and populate rows
    wb = openpyxl.Workbook()
    ws = wb.active
    ws.title = "Validation"

    ws.append(["filename", "directory", "score(1-5)", "notes"])

    for fname, dname in selected_files:
        ws.append([fname, dname, "", ""])

    wb.save(output_file)
    print(f"Saved {len(selected_files)} files to: {output_file}")


def select_speakers_for_partitions(path_info_speakers, dict_speakers, seed=None):
    """
    - Selects speakers (~20%) that are going to appear only in test set (preventing voice leakages across shared speakers & multi-speaker authors).
    - Gives a list of the remaining speakers that will be shared by training and development set.
    - Saves/loads to CSV lists of speakers.
    - Prints duration summary.
    """
    df = pd.read_html(path_info_speakers)[0]
    csv_path = Path(path_info_speakers).parent / "speakers_in_partitions.csv"

    if csv_path.exists():
        test_spks = pd.read_csv(csv_path).query("partition == 'test'")[
            "speaker"
        ].tolist()
    else:
        graph = defaultdict(set)
        for raw in dict_speakers.values():
            spks = [s.strip() for s in raw.split("/") if s.strip()]
            for i in range(len(spks)):
                for j in range(i + 1, len(spks)):
                    graph[spks[i]].add(spks[j])
                    graph[spks[j]].add(spks[i])


        all_spks = sorted(df["Speakers"].unique()) # Extract fully connected speaker components (clusters)
        visited = set()
        clusters = []

        for spk in all_spks:
            if spk not in visited:
                component = []
                queue = [spk]
                while queue:
                    curr = queue.pop()
                    if curr not in visited:
                        visited.add(curr)
                        component.append(curr)
                        # Sort neighbor set so traversal order is 100% deterministic
                        queue.extend(sorted(graph[curr] - visited))
                clusters.append(sorted(component))

        # Sort clusters deterministically by their first speaker
        clusters.sort(key=lambda c: c[0])

        # Sum audio times per cluster
        spk_time = dict(zip(df["Speakers"], df["Audio time (s)"]))
        cluster_data = [
            (c, sum(spk_time.get(s, 0.0) for s in c)) for c in clusters
        ]
        target = df["Audio time (s)"].sum() * 0.20

        # Sum audio times per cluster
        spk_time = dict(zip(df["Speakers"], df["Audio time (s)"]))
        cluster_data = [
            (c, sum(spk_time.get(s, 0.0) for s in c)) for c in clusters
        ]
        target = df["Audio time (s)"].sum() * 0.20

        # Sample clusters closest to target duration
        if seed is not None:
            random.seed(seed)

        best = min(
            (
                random.sample(
                    cluster_data,
                    random.randint(1, max(1, len(cluster_data) // 2)),
                )
                for _ in range(10000)
            ),
            key=lambda sample: abs(sum(dur for _, dur in sample) - target),
        )

        test_spks = sorted([spk for cluster, _ in best for spk in cluster])

        # Save to CSV
        df.assign(
            partition=df["Speakers"].apply(
                lambda s: "test" if s in test_spks else "train_dev"
            )
        )[["Speakers", "partition"]].rename(
            columns={"Speakers": "speaker"}
        ).to_csv(csv_path, index=False)
        print(
            "A csv file called 'speakers_in_partitions.csv' with the speakers selected for each partition was saved.\n"
        )
    train_dev_spks = df[~df["Speakers"].isin(test_spks)]["Speakers"].tolist()
    return train_dev_spks, test_spks


def create_partitions(test_speakers, dict_speakers, dir_chunks_1, dir_chunks_2, dataset_split_path, seed=42):
    """
    Distributes Whisper WAV/TXT chunks into train, dev, and test directories,
    ensuring 1:1 partition matching across both short and long datasets.
    """
    if seed is not None:
        random.seed(seed)

    split_path = Path(dataset_split_path)
    test_spks_set = set(test_speakers)

    # Clean destination directories
    for partition in ["train", "dev", "test"]:
        part_dir = split_path / partition
        if part_dir.exists():
            shutil.rmtree(part_dir)
        part_dir.mkdir(parents=True, exist_ok=True)

    # Collect and sort all files deterministically
    wav_files = sorted(
        list(Path(dir_chunks_1).glob("*.wav")) + list(Path(dir_chunks_2).glob("*.wav")),
        key=lambda p: p.name
    )

    story_partition_map = {} # Cache assignment per recording story so short and long chunks follow identical rules

    for wav in wav_files:
        # Extract author code (e.g., 'rgn_bal_1...' -> 'bal')
        m_author = re.search(r"rgn_([a-z]{3,4})_", wav.name)
        if not m_author:
            continue
        author = m_author.group(1)
        story_id = re.sub(r"_(seg|chunk)\d+.*$", "", wav.stem)  # Extract base story prefix (e.g. 'rgn_bal_1_seg001' -> 'rgn_bal_1', 'rgn_bal_1_chunk001' -> 'rgn_bal_1')
        spks = [s.strip() for s in dict_speakers.get(author, "").split("/") if s.strip()]
        
        if any(spk in test_spks_set for spk in spks): # Determine partition
            target_partition = "test"
        else:
            if story_id not in story_partition_map:
                story_partition_map[story_id] = "train" if random.random() < 0.875 else "dev"
            target_partition = story_partition_map[story_id]

        target = split_path / target_partition

        shutil.copy2(wav, target / wav.name)  # Copy WAV and paired TXT
        txt = wav.with_suffix(".txt")
        if txt.exists():
            shutil.copy2(txt, target / txt.name)

    print(f"Successfully created train, dev, and test partitions in: {split_path}")

### 0.3 Global variables

In [ ]:
# Romagnol-Italian dataset
SHARED_DATASET_DIR = Path.home() / "OneDrive - Alma Mater Studiorum Università di Bologna" / "Debora Ciminari's files - tesi_Irene" / "data"

# Dedicated directory to save all output from this script.
OUTPUT_DIR = Path.home() / "OneDrive - Alma Mater Studiorum Università di Bologna" / "Master's Dissertation" / "Pre-processing_output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Alternatively, switch to this directory to create the output folder inside Documents.
# OUTPUT_DIR = Path.home() / "Downloads" / "Pre-processing_output"
# OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

STATISTICS_DATA = OUTPUT_DIR / "statistics_data_graphs"
STATISTICS_DATA.mkdir(parents=True, exist_ok=True)

RESOURCES_RGN_SUMMARY = STATISTICS_DATA / "risorse_rgn"

FILES_TO_VALIDATE = OUTPUT_DIR / "files_to_validate.xlsx"
STATISTICS_DATA.mkdir(parents=True, exist_ok=True)

SHORTSEG_DATASET = OUTPUT_DIR / "shortseg_dataset_split"
SHORTSEG_DATASET.mkdir(parents=True, exist_ok=True)

LONGSEG_DATASET = OUTPUT_DIR / "longseg_dataset_split"
LONGSEG_DATASET.mkdir(parents=True, exist_ok=True)

train_set_shortseg = SHORTSEG_DATASET / "train"
dev_set_shortseg = SHORTSEG_DATASET / "dev"
test_set_shortseg = SHORTSEG_DATASET / "test"

train_set_longseg = LONGSEG_DATASET / "train"
dev_set_longseg = LONGSEG_DATASET / "dev"
test_set_longseg = LONGSEG_DATASET / "test"

# Test audio
TEST_AUDIO_1 = OUTPUT_DIR / "rgn_only" / "audio" / "rgn_bal_1.wav"
TEST_TEXT_1 = OUTPUT_DIR / "rgn_only" / "text" / "rgn_bal_1.txt"

TEST_AUDIO_2 = OUTPUT_DIR / "rgn-ita" / "audio_split" / "rgn_bal_1_st1.wav"
TEST_TEXT_2 = OUTPUT_DIR / "rgn-ita" / "stories_split" / "rgn_bal_1_st1.txt"

# Project subdirectories for audio, text and whisper Chunks.

rgnonly_audio = OUTPUT_DIR / "rgn_only/audio"
rgnonly_text = OUTPUT_DIR / "rgn_only/text"
rgnonly_chunks = OUTPUT_DIR / "rgn_only/whisper_chunks"
rgnonly_longer_chunks = OUTPUT_DIR / "rgn_only/longer_whisper_chunks"

rgnita_audio = OUTPUT_DIR / "rgn-ita/audio_split"
rgnita_text = OUTPUT_DIR / "rgn-ita/stories_split"
rgnita_chunks = OUTPUT_DIR / "rgn-ita/whisper_chunks"
rgnita_longer_chunks = OUTPUT_DIR / "rgn-ita/longer_whisper_chunks"

DATASETS = {
    "rgnonly": (rgnonly_audio, rgnonly_text, rgnonly_chunks),
    "rgnita": (rgnita_audio, rgnita_text, rgnita_chunks),
}

In [ ]:
dict_speakers = {
    "bal": "sirri",
    "bald": "baldini/piccari/redeghieri",
    "bar": "bartoli",
    "bel": "belletti",
    "bell": "bellosi",
    "bon": "bonoli",
    "bor": "borgini",
    "cas": "casadei",
    "fab": "fabbri",
    "fac": "facciani/bianco",
    "far": "farneti",
    "fos": "foschi",
    "fuc": "fucci",
    "gab": "gabellini",
    "gal": "galli/campitelli",
    "gio": "gori",
    "gor": "gori",
    "gue": "guerra",
    "guer": "bellosi",
    "lor": "lorenzi",
    "lug": "lugaresi",
    "mal": "campitelli",
    "maz": "mazzotti/ferorelli",
    "mol": "molari",
    "nad": "nadiani/redeghieri",
    "nar": "nardini",
    "nos": "nostri",
    "paz": "pazzini",
    "ped": "pedrelli",
    "pedr": "baldini/guerra",
    "pol": "polini",
    "roc": "teodorani",
    "ros": "rossini",
    "spa": "spadoni",
    "sta": "stargiotti",
    "teo": "teodorani",
    "tur": "turci",
    "zan": "zanotti"
}

## 1. Text pre-processing <br>

Pipeline for *Text files in Romagnol only* and *Parallel text files in Romagnol and Italian*:
1. Access original shared dataset (Debora's OneDrive folder)
2. Access sub-folder "data"
3. Access sub-folders "rgn_only" and "rgn-ita"
4. In each of these sub-folders, access "text"
5. For each file in "text" apply normalization strategies (Lowercase, Remove punctuation, Diacritic Normalization)
6. (only for *Parallel text files in Romagnol and Italian*) For each file, split into different stories and save each story in a standalone file.
7. Removing titles from 41 transcripts when it is not said in the recording (list was created during validation of aligned+segmented chunks)

### 1.1 Text normalization

In [ ]:
# Folders to process i.e. to read, lowercase, strip of punctuation, and save to the output folder with their original names intact.
preprocess_text(input_dir=SHARED_DATASET_DIR, text_folders=["rgn_only/text", "rgn-ita/text"], output_dir=OUTPUT_DIR)

### 1.2 Splitting different stories contained in the same file in the rgn-ita folder

In [ ]:
split_text_stories(input_dir=OUTPUT_DIR, name_input_folder="rgn-ita/text", output_dir=OUTPUT_DIR, name_output_folder="rgn-ita/stories_split")

### 1.3 Removing titles from transcript when it is not said in the recording

In [ ]:
st_title_unsaid_rgnonly = ["rgn_gal_20"]
st_title_unsaid_rgnita = [
    "rgn_bald_2_st1", "rgn_bald_2_st3", "rgn_bald_2_st5", "rgn_bald_2_st6",
    "rgn_bell_2_st1", "rgn_bell_2_st10", "rgn_bell_2_st2",
    "rgn_bell_2_st3", "rgn_bell_2_st5", "rgn_bell_2_st6",
    "rgn_bell_2_st7", "rgn_bell_2_st9", "rgn_gab_2_st5",
    "rgn_gab_2_st6", "rgn_gab_2_st7", "rgn_gal_1_st10",
    "rgn_gor_2_st10", "rgn_gor_2_st11", "rgn_gue_4_st1",
    "rgn_gue_4_st3", "rgn_gue_4_st4", "rgn_mal_1_st1",
    "rgn_mal_1_st10", "rgn_mal_1_st2", "rgn_mal_1_st3",
    "rgn_mal_1_st4", "rgn_mal_1_st5", "rgn_mal_1_st6",
    "rgn_mal_1_st7", "rgn_mal_1_st8", "rgn_mal_1_st9",
    "rgn_pedr_1_st7", "rgn_pol_1_st1", "rgn_pol_1_st2",
    "rgn_pol_1_st3", "rgn_pol_1_st4", "rgn_pol_1_st5", "rgn_pol_1_st6",
    "rgn_pol_1_st7", "rgn_spa_1_st10", "rgn_spa_1_st11", "rgn_gue_4_st10"
]
print(len(st_title_unsaid_rgnonly))
print(len(st_title_unsaid_rgnita))

In [ ]:
# Run this cell only once to remove the first two lines from the transcripts in the lists above.
remove_titles(list_of_files=st_title_unsaid_rgnonly, text_dir=rgnonly_text)
remove_titles(list_of_files=st_title_unsaid_rgnita, text_dir=rgnita_text)

### 1.4 Checking total number of stories (transcripts in txt format)

In [ ]:
print_stats(OUTPUT_DIR, "rgn-ita/stories_split", "ita", "rgn")
print_stats (OUTPUT_DIR, "rgn_only/text")

## 2. Audio pre-processing <br>

Recent low-resource and dialectal ASR studies using Whisper or Wav2Vec2/XLS-R architectures (such as Pillai et al., 2026 for Malasar and Tsoukala et al., 2026 for Greek dialects) rely on a simple audio pre-processing pipeline, in which audio is standardized to 16 kHz mono WAV, amplitude is normalized, and then audio is segmented into utterances under 30 seconds before fine-tuning.

(Detailed) Steps in this pre-processing pipeline:

1. Loading original MP3 files.
2. Resampling: all input recordings are standardized to 16,000 Hz, mono (this is the standard input format requirement for Whisper speech models).
3. Low-pass filter: this filter is applied so as to remove high-frequency microphone hiss/buzz. Human speech sampled at 16 kHz contains critical high-frequency acoustic information up to the 8000 Hz limit, so the cutoff frequency was set at 7000 Hz. Almost all natural speech energy should be preserved with a cutoff point like this, while still removing high-frequency digital artifacts and equipment noise.
4. Amplitude Peak Normalization: this step is meant to ensure that recordings across different authors and microphone setups have standardized volume levels (aka harmonize volume disparities). It is done by scaling the wave peak to a consistent value (defaulting to 0.95 / 95% of maximum amplitude). This proved to lower lower character error rates (CER).
5. Save WAV files.

(For me to remember) N.B. Models like Whisper convert the raw audio waveform into 80-channel log-magnitude Mel spectrograms using 25 ms windows with a 10 ms stride and apply internal layer normalization / zero-mean unit-variance scaling before processing the input.

(For rgn-ita stories): <br>

6. Manual Audacity Labeling: The master .wav files are segmented into the different Romagnol recorded stories they contain using Audacity. Music transitions, long silences, and story changes are visible on the waveform. Regions of the .wav files where stories were told in Italian were overlooked. 
> Manual splitting and labeling is best in this project because:
- It allowed me to have as little dataset noise as possible by having control over the boundaries for each story/segment. Automated tools (silence detection, VAD) will occasionally provide a faulty automatic segmentation by cutting off the first syllable of a poem, clipping soft trailing words, or leaking several seconds of guitar/music into a speech segment.
- I was able to match filenames exactly (e.g., rgn_bal_1_st1.wav to rgn_bal_1_st1.txt), and to work with a system for multi-part stories and then be able to merge them all together: a single story could include multiple musical intermezzos, so a situation like this could have happened rgn_bal_1_st1a.wav, rgn_bal_1_st1b.wav, rgn_bal_1_st1c (single story divided into 2 parts).
>>> Hope is to have CET/WER benchmarks measure pure acoustic and language model performance without error propagation from low-quality data. <br>

7. Cleaning filenames + merging multi-part stories: two function were run after manual segmentation on Audacity to: 1. get rid on any possible whitespace in filenames for each Romagnol story in the rgn-ita folder, 2. merge multi-part stories (e.g. rgn_bal_1_st1a, rgn_bal_1_st1b, rgn_bal_1_st1c were merged into rgn_bal_1_st1).

### 2.1 Audio resampling and normalization

In [ ]:
preprocess_audio(input_dir=SHARED_DATASET_DIR, input_folders=("rgn_only/audio", "rgn-ita/audio"), output_dir=OUTPUT_DIR)

### 2.2 Manual segmentation on Audacity

### 2.3 Cleaning filenames + Merging multi-part stories

In [ ]:
clean_filenames(rgnita_audio)
clean_filenames(rgnonly_audio)

In [ ]:
merge_multi_part_audio(rgnita_audio)

### 2.4 Checking total number of audio stories (in wav format)

In [ ]:
print_stats(OUTPUT_DIR, "rgn-ita/audio_split", "rgn")
print_stats(OUTPUT_DIR, "rgn_only/audio")

## 3. STT (Speech-To-Text) Forced alignment and segmentation <br>

After manual labeling in Audacity, here is the breakdown of the dataset:
1. Files over 30 s in 'rgn_only\audio' and 'rgn-ita\audio_split': 201 out of 350 (57%).
2. Total audio time in directory 'rgn_only\audio' + 'rgn-ita\audio_split': 4.60 hours (16561 seconds).
3. Mean audio time and standard deviation: 47.32 ± 46.51.

Next up: STT (Speech-To-Text) Forced alignment. Faster Whisper combined with Stable Whisper is used for word- and line-level forced alignment to generate word-level timestamps, and possible audio-text aligned segments with corresponding start and end timestamps that can be used to slice both audio and text. It does so by leveraging Whisper's Native cross-attention (Dynamic Time Warping), natural silences and line boundaries in the story transcripts.
> It is necessary because each audio file must correspond 1:1 with its ground-truth text transcript (transcript for an audio recording should contain only the exact words spoken in that specific slice) + words pronounced in in an audio segment should not be randomly cut.

Following: Segmentation. In order to use the audio files to fine-tune Whisper models, it is necessary to split them into slices of max 30 seconds. The timestamps generated with the forced-alignment are used to create audio and text chunks. <br>

N.B. Boundary Padding Buffers are also added. Currently, 0.18 seconds (180 milliseconds) of padding are added before the start of the speech segment (- 0.18) and 0.20 seconds (200 milliseconds) of padding after the end of the speech segment (+ 0.20). This guarantees that: <br>
- initial plosives and word-final syncopated consonants (subtle acoustic tails common in Romangol Romagnol e.g. sbdêl or strmèn) are not clipped at the edges
- any under-estimated start or end timestamps, especially on fast speech, are never cut off

>>> (For me to remember) Stable-ts and faster-whisper work together as a two-layer stack. Faster-whisper acts as the low-level compute engine(= it takes the Whisper weights (quantized to int8 or float16) and runs the forward passes through the Mel-filterbank, the Audio Encoder, and the Text Decoder), while stable-ts acts as the mathematical orchestrator that re-routes how that engine executes (= it calls faster-whisper's encoder and decoder in a forced-alignment mode rather than a transcription mode).

***Validation step***  <br> 
After alignment and segmentation, 58 randomly selected files, lasting 41 minutes and 16.73 seconds, corresponding to 15% of the total audio time in the dataset, are manually validated. The human validator checks all Whisper chunks generated for each story (by listening to the audio clips and their transcripts), and assigns a score on a scale from 1-5 on a dedicated excel file.  <br>

> 1 = poor [3+ major errors (+ minor errors)]<br>
> 2 = below expectations [2 major errors (+ minor errors)]<br>
> 3 = acceptable [1 major error (+ minor errors)] <br>
> 4 = good [only minor errors] <br>
> 5 = excellent [no errors] <br>

What counts as **minor** error: **(1) missed initial sound** or **(2) missed final sound**, in one or more segmented audio clip. <br>
What counts as **major** error: **(1) extra/ missing words in audio** or **(2) extra/missing words in transcript** or **(3) transcript and audio not matching at all**, in one or more segmented audio clip. <br>

Merging segments 

### 3.1 Getting models ready

In [ ]:
# Detect hardware
is_cuda, device, compute_type, model_size = env_check()

# Resolve path/identifier (downloads locally if on CPU and missing)
model_dir = setup_model(model_size, ".") # on laptop model_id is path to downloaded weights. On server, it's "large_v3"

In [ ]:
stable_whisper_model = load_stable_whisper(model_dir, device, compute_type)

### 3.2 Creating Whisper chunks (STT alignment and segmentation)

Parameters used in model.align():
- suppress_silence=False <br>
Why: Setting this to True masks out low-energy frames before the attention matrix is computed. Regional speech frequently has quiet phrase endings, trailing vowels, and natural inhalation pauses. If silence suppression is active, low-amplitude word endings get zeroed out, causing Dynamic Time Warping (DTW) to stall and report trailing unaligned tokens. Keeping it False preserves the entire acoustic timeline.
- vad=False <br>
Why: The built-in Silero VAD is trained predominantly on standard, clean speech. Regional phonology (such as geminate consonants, apheresis, or rapid unvoiced fricatives) often confuses generic VAD models into flagging valid speech as background noise. Setting it to False prevents premature cuts of quiet word onsets and tails.
- original_split=True <br>
Why: Forces stable-whisper to retain your exact line breaks from the input text rather than letting Whisper re-segment the output based on arbitrary punctuation or token count heuristics. This guarantees a 1-to-1 match between your text lines and the resulting segment objects.
- failure_threshold=None <br>
Why: By default, failure_threshold acts as a circuit breaker (typically set around 0.4–0.5). When standard Italian cross-attention encounters dialect clitics and non-standard orthography, confidence scores naturally dip. A numeric threshold causes the aligner to give up midway and collapse the remaining segments. Setting it to None disables early termination, forcing the DTW path to continue through to the end of the file.
- fast_mode=False <br>
Why: fast_mode=True prunes the search space to a narrow corridor around the diagonal. In dialect speech, where speaking rates shift or unexpected pauses occur, the true acoustic path easily steps outside this narrow band. Disabling fast mode computes the complete global alignment matrix without pruning, preventing path divergence.
- word_dur_factor=2.0 (default is 1.0 or 1.5) <br>
Why: Dictates the maximum expected duration for any single word before the search algorithm assumes it is off-track. Dialect authors often draw out vowels expressively or pause mid-clause. Increasing this multiplier to 2.0 relaxes the duration boundaries, accommodating expressive cadences without failing word units.

In [ ]:
# Run this cell to know how many stories are longer than e.g. 15 seconds (before segmentation).
info_durations(dir_1=rgnonly_audio, dir_2=rgnita_audio, output_dir=STATISTICS_DATA, over_time=15)

In [ ]:
# Run this cell to align and segment the stories in rgn_only folder. 
align_and_chunk(stable_whisper_model, "rgnonly")

In [ ]:
# Run this cell to align and segment the stories in rgn-ita folder.
align_and_chunk(stable_whisper_model, "rgnita")

In [ ]:
# Run this cell to know how many stories are longer than e.g. 15 seconds (after segmentation).
info_durations(dir_1=rgnonly_chunks, dir_2=rgnita_chunks, output_dir=STATISTICS_DATA, over_time=15)

### 3.3 Validation step
1. Checking and retrying align_and_chunk() on single files that gave issues during alignment + segmentation step.
2. Selecting files (amounting to 15% of the total time) whose segmentation will be manually evaluated.

In [ ]:
# Use this cell to align and segment a single file in the rgn-ita folder.
align_and_chunk(stable_whisper_model, "rgnita", single_file="rgn_gue_4_st9")

In [ ]:
# Use this cell to align and segment a single file in the rgn_only folder.
align_and_chunk(stable_whisper_model, "rgnonly", single_file="rgn_gal_2")

In [ ]:
# Run this cell to select the files that will be manually evaluated.
files_to_validate(dir1=rgnonly_audio, dir2=rgnita_audio, percentage=15, output_path=FILES_TO_VALIDATE)

In [ ]:
# Free model memory after alignment and segmentation
del stable_whisper_model # Delete model variable
gc.collect()  # Force Python garbage collector to free CPU RAM memory

if is_cuda:
    torch.cuda.empty_cache()    # Free GPU memory if running on CUDA

### 3.4 Creating longer Whisper chunks
= Merging adjacent chunks (up to 10–30 seconds) <br>

So far, two directories, both called Whisper chunks were created. Each contains pairs of aligned txt and wav files (a transcript and an audio clip for each pair). These are short segments (numbered sequentially) of longer recordings.<br>
These are examples of what the filenames can look like inside the partitions: rgn_bal_1_seg001, rgn_bal_1_seg006, rgn_bal_2_seg019, rgn_gal_19_seg004, rgn_tur_2_st9_seg013, rgn_tur_2_st5_seg008, rgn_gor_2_st9_seg001, etc. <br>
Since the average duration of these audio clips is very small (2.5 seconds), I need to concatenate consecutive segments to provide better context to the ASR models.

In [ ]:
def merge_whisper_chunks(input_dir, output_dir, target_seconds=15.0, max_seconds=28.5, silence_padding_ms=150):
    os.makedirs(output_dir, exist_ok=True)

    # Gather all matching .wav and .txt pairs
    groups = defaultdict(list)
    for wav_path in Path(input_dir).glob("*.wav"):
        txt_path = wav_path.with_suffix(".txt")
        if not txt_path.exists():
            continue
        match = re.match(r"^(.*)_seg(\d+)$", wav_path.stem)
        if match:
            groups[match.group(1)].append((int(match.group(2)), wav_path, txt_path))

    # Process each recording group
    for prefix, files in tqdm(groups.items(), desc="Merging Chunks", unit="rec"):
        files.sort(key=lambda x: x[0])
        
        # Partition segments into merged chunks
        chunks = []
        batch, batch_dur = [], 0.0

        for seg_num, wav_p, txt_p in files:
            with wave.open(str(wav_p), "rb") as w:
                dur = w.getnframes() / float(w.getframerate())

            added_silence = (silence_padding_ms / 1000.0) if batch else 0.0

            # If gap detected, target reached, or next segment exceeds max duration -> split
            if batch and (seg_num != batch[-1][0] + 1 or batch_dur >= target_seconds or batch_dur + added_silence + dur > max_seconds):
                chunks.append(batch)
                batch, batch_dur, added_silence = [], 0.0, 0.0

            batch.append((seg_num, wav_p, txt_p))
            batch_dur += added_silence + dur

        if batch:
            chunks.append(batch)

        # Write each chunk to disk (single write implementation)
        for idx, chunk in enumerate(chunks, start=1):
            with wave.open(str(chunk[0][1]), "rb") as f_w:
                params = f_w.getparams()

            silence_frames = int(params.framerate * (silence_padding_ms / 1000.0))
            silence_bytes = b"\x00" * (silence_frames * params.nchannels * params.sampwidth)

            # Write merged audio
            out_wav = os.path.join(output_dir, f"{prefix}_chunk{idx:03d}.wav")
            with wave.open(out_wav, "wb") as o_w:
                o_w.setparams(params)
                for i, (_, wav_p, _) in enumerate(chunk):
                    if i > 0 and silence_padding_ms > 0:
                        o_w.writeframes(silence_bytes)
                    with wave.open(str(wav_p), "rb") as in_w:
                        o_w.writeframes(in_w.readframes(in_w.getnframes()))

            # Write merged text
            texts = [Path(txt_p).read_text(encoding="utf-8").strip() for _, _, txt_p in chunk]
            out_txt = os.path.join(output_dir, f"{prefix}_chunk{idx:03d}.txt")
            Path(out_txt).write_text(" ".join(t for t in texts if t), encoding="utf-8")

In [ ]:
# Process rgn_only folder
merge_whisper_chunks(input_dir=rgnonly_chunks, output_dir=rgnonly_longer_chunks, target_seconds=15.0)

In [ ]:
# Process rgn-ita folder
merge_whisper_chunks(input_dir=rgnita_chunks, output_dir=rgnita_longer_chunks, target_seconds=15.0)

## 4. Splitting dataset

Firstly, you can calculate each author and then each speaker's contribution in the dataset, so as to better visualize how present and represented they are. <br>

Target : Training 70%, Development 10%, Test: 20%. <br>
**N.B.** Data in test partition need to be completely new for the model (files from speakers unseen during training and development).

> During the selection, I made sure that in cases like 1 and 2, the speakers and the stories they gave voice to all went into the same partition.
> 1. speakers reading stories from multiple authors e.g. Gori reads both stories by "gio" and "gor".
> 2. multiple speakers reading the stories by the same authors e.g. Baldini/Piccari/Redeghieri all reading stories by "bald"

Partitioning Strategy: <br> 

1. Isolated Test Set (Speaker Level) <br>
Select a subset of distinct speakers whose combined duration total is as close to 20% as possible. These speakers and all their files are assigned exclusively to Test.

2. Train & Dev Sets (Story/File Level) <br>
Remaining pool: 80% of total audio time across the remaining speakers. We distribute individual stories/files from these speakers into Train (87.5%) and Dev (12.5%). This allows speakers to appear in both Train and Dev while ensuring Test remains 100% unseen. <br>
N.B. The custom function create_partitions() works deterministically so that when this script is run on the same machine the results will always be the same and the same stories will always end up in the train set and dev set (this ensures that the two datasets I am working with )

These are the assumptions before creating the dataset partitions: <br>
- All wav clips and paired txt files contain one author name (3 or 4 letters) in their filenames.
- dict_speakers maps that author name directly to their speaker IDs.
- There are two distinct directories storing the short Whisper chunks (.wav and .txt files with avg time 3s) and longer Whisper chunks (merged .wav and .txt files with avg time 15s).
- .wav and .txt pairs share the exact same stem names.

### 4.1 Calculating each author's contribution to the dataset (speech time, tokens, number of stories)
N.B. In html file, authors are sorted alphabetically in a descending order (a-z).

In [ ]:
info_authors(dir_audio_1=rgnonly_audio, dir_audio_2=rgnita_audio,
              dir_txt_1=rgnonly_text, dir_txt_2=rgnita_text,
              output_dir=STATISTICS_DATA,
              totals=True,
              pie=True,
              bar=True
              )

### 4.2 Calculating each speaker's contribution to the dataset (speech time, tokens, number of stories)
N.B. In the htlm file, speakers are sorted in descending order based on their total audio time.

In [ ]:
info_speakers(dir_audio_1=rgnonly_audio, dir_audio_2=rgnita_audio,
              dir_txt_1=rgnonly_text, dir_txt_2=rgnita_text,
              output_dir=STATISTICS_DATA,
              dict_speakers=dict_speakers,
              totals=True,
              pie=True,
              bar=True
              )

### 4.3 Selecting speakers going into different dataset partitions

In [ ]:
train_dev_spks, test_spks = select_speakers_for_partitions(
    path_info_speakers=STATISTICS_DATA / "info_speakers" / "info_speakers_df.html", dict_speakers=dict_speakers, seed=42)

print("Test Speakers:", test_spks)
print("Train + Dev Speakers (Shared):", train_dev_spks)

### 4.4 Dividing Whisper chunks into three partitions

In [ ]:
# Creating shortseg dataset (containing shorter segments with avg 3.15 s)
create_partitions(
    test_speakers=test_spks,
    dict_speakers=dict_speakers, 
    dir_chunks_1=rgnita_chunks, 
    dir_chunks_2=rgnonly_chunks, 
    dataset_split_path=SHORTSEG_DATASET,
    )

In [ ]:
# Creating longseg dataset (containing longer segments with 15)
create_partitions(
    test_speakers=test_spks,
    dict_speakers=dict_speakers, 
    dir_chunks_1=rgnita_longer_chunks, 
    dir_chunks_2=rgnonly_longer_chunks, 
    dataset_split_path=LONGSEG_DATASET,
    )

## 5. Data summary <br>

Dataset description:

Duration: 
* Raw audio duration
* Audio duration once files are all cleaned and segmented (hours, minutes, seconds) (both versions of the dataset i.e. shortseg and longseg)
* Net speech time vs. total audio time (measuring silence/pause proportions).
* Mean, median, standard deviation in shortseg and longsed datasets. 

Author and speaker:

- Summary of author and speaker for each file.
- Total contribution per speaker (speech time, tokens, number of stories) 
- Total contribution per author (audio time of his/her stories, tokens, number of stories)


In [ ]:
export_file_speaker_mapping(dir_rgnonly_chunks=rgnonly_chunks, dir_rgnita_chunks=rgnita_chunks,
                            dict_speakers=dict_speakers, output_excel_path=RESOURCES_RGN_SUMMARY)

### 5.1 Calculating raw audio duration

In [ ]:
info_durations(dir_1=rgnita_audio, dir_2=rgnonly_audio, output_dir=STATISTICS_DATA, overview=True, graph=True, totals=True)

In [ ]:
info_durations(dir_1=rgnonly_audio, output_dir=STATISTICS_DATA, overview=True, totals=True, graph=True)

In [ ]:
info_durations(dir_1=rgnita_audio, output_dir=STATISTICS_DATA, overview=True, totals=True, graph=True)

### 5.2 Calculating segmented audio chunks' duration
Note: the total audio time is lower now because room noise and silence gaps were removed. <br>
The pipeline is working as intended: the ASR model trains strictly on speech.
- When silence and background noise are removed across a whole dataset, trimmed time tends to scale proportionally across speakers.

#### 5.2.1 Short Whisper chunks

In [ ]:
info_durations(dir_1=rgnonly_chunks, output_dir=STATISTICS_DATA, overview=True,
               totals=True, graph=True, dir_2=rgnita_chunks)

In [ ]:
info_durations(dir_1=rgnonly_chunks, output_dir=STATISTICS_DATA, overview=True, totals=True, graph=True)

In [ ]:
info_durations(dir_1=rgnita_chunks, output_dir=STATISTICS_DATA, overview=True, totals=True, graph=True)

#### 5.2.2 Longer Whisper chunks
Note: the total audio time is lower now because room noise and silence gaps were removed. <br>
The pipeline is working as intended: the ASR model trains strictly on speech.
- When silence and background noise are removed across a whole dataset, trimmed time tends to scale proportionally across speakers.

In [ ]:
info_durations(dir_1=rgnonly_longer_chunks, output_dir=STATISTICS_DATA, overview=True,
               totals=True, graph=True, dir_2=rgnita_longer_chunks)

In [ ]:
info_durations(dir_1=rgnonly_longer_chunks, output_dir=STATISTICS_DATA, overview=True, totals=True, graph=True)

In [ ]:
info_durations(dir_1=rgnita_longer_chunks, output_dir=STATISTICS_DATA, overview=True, totals=True, graph=True)

### 5.3 Calculating info about train, dev and test partitions 
| Partition|Shortseg Duration|Longseg Duration|Shortseg %|Longseg %|Added Silence Padding (∼150 ms/joint)|Shortseg Clip Length (μ±σ)|Longseg Clip Length (μ±σ)|
| ----------|-------------------|------------------|----------|-----|-----------------------------------|----------------------|---------------------|
|Train|10,381 s (2.88 h)|10,786 s (3.00 h)|70.88%|70.81%|+405 s|3.26 ± 2.93 s|15.23 ± 4.99 s|
|Dev|1,378 s (0.38 h)|1,450 s (0.40 h)|9.41%|9.52%|+72 s|2.42 ± 1.73 s|14.64 ± 4.92 s|
|Test|2,886 s (0.80 h)|2,996 s (0.83 h)|19.71%|19.67%|+110 s|3.30 ± 2.42 s|14.54 ± 5.28 s|
|Total Split|14,645 s (4.07 h)|15,232 s (4.23 h)|100.00%|100.00%|+587 s|—|—|
|Raw Reference|16,512 s (4.59 h)|16,512 s (4.59 h)|—|—|—|—|—|

#### 5.3.1 Shortseg dataset

In [ ]:
info_durations(dir_1=train_set_shortseg, output_dir=STATISTICS_DATA, totals=True, graph=True)

In [ ]:
info_durations(dir_1=dev_set_shortseg, output_dir=STATISTICS_DATA, totals=True, graph=True)

In [ ]:
info_durations(dir_1=test_set_shortseg, output_dir=STATISTICS_DATA, totals=True, graph=True)

#### 5.3.1 Longseg dataset

In [ ]:
info_durations(dir_1=train_set_longseg, output_dir=STATISTICS_DATA, totals=True, graph=True)

In [ ]:
info_durations(dir_1=dev_set_longseg, output_dir=STATISTICS_DATA, totals=True, graph=True)

In [ ]:
info_durations(dir_1=test_set_longseg, output_dir=STATISTICS_DATA, totals=True, graph=True)